# 🤝 Multi-Model Stacking & Blending Ensemble (Kaggle PGS S6E11)

Welcome to the **Ensemble Pipeline** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 🎯 Objectives
1. Implement **5-Fold Stratified Cross-Validation** simultaneously across all four established model families:
   - **Shallow LightGBM** (`num_leaves=4`, `max_depth=3`)
   - **Shallow XGBoost GPU** (`max_depth=3`, `tree_method='hist'`)
   - **CatBoost GPU** (`depth=4`, symmetric trees)
   - **Regularized Logistic Regression** (StandardScaler + OneHotEncoder)
2. Ensure **100% fold alignment** using `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
3. Compute **Spearman Rank Correlation** to confirm model prediction diversity.
4. Compare multiple ensembling strategies:
   - **Strategy A**: Rank-Averaged Ensemble
   - **Strategy B**: Nelder-Mead Constrained Weighted Blend (optimizing ROC-AUC directly)
   - **Strategy C**: Logistic Regression Stacking Meta-Learner
5. Generate and rigorously verify the final ensemble `submission.csv`.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import rankdata

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Hardware Acceleration Device: {device}")
print(f"LightGBM: {lgb.__version__} | XGBoost: {xgb.__version__}")

In [ ]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")

In [ ]:
print("Loading datasets...")
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

features = [c for c in train_df.columns if c not in ['id', 'is_fraud']]
cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
num_cols = [c for c in features if c not in cat_cols]
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

print(f"Train rows: {len(train_df):,}, Test rows: {len(test_df):,}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features ({len(num_cols)}): {num_cols}")

In [ ]:
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

# Allocate OOF and Test prediction arrays for all 4 models
n_train = len(train_df)
n_test = len(test_df)

oof_lgb = np.zeros(n_train, dtype=np.float64)
oof_xgb = np.zeros(n_train, dtype=np.float64)
oof_cat = np.zeros(n_train, dtype=np.float64)
oof_lr  = np.zeros(n_train, dtype=np.float64)

test_lgb = np.zeros(n_test, dtype=np.float64)
test_xgb = np.zeros(n_test, dtype=np.float64)
test_cat = np.zeros(n_test, dtype=np.float64)
test_lr  = np.zeros(n_test, dtype=np.float64)

y = train_df['is_fraud'].astype(int).values

# Prepare data representations
# 1. Categorical formatted for GBDT
X_gbdt = train_df[features].copy()
X_test_gbdt = test_df[features].copy()
for c in cat_cols:
    X_gbdt[c] = X_gbdt[c].astype('category')
    X_test_gbdt[c] = X_test_gbdt[c].astype('category')
for b in bool_cols:
    X_gbdt[b] = X_gbdt[b].astype(int)
    X_test_gbdt[b] = X_test_gbdt[b].astype(int)

# 2. Preprocessor pipeline for Logistic Regression
lr_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

print("=" * 70)
print("Starting 5-Fold Stratified Cross-Validation for 4 Base Models...")
print("=" * 70)

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y), 1):
    fold_start = time.time()
    print(f"\n--- Fold {fold}/{Config.N_SPLITS} ---")
    
    # ------------------ 1. Logistic Regression ------------------
    X_tr_lr = lr_preprocessor.fit_transform(train_df.iloc[train_idx][features])
    X_va_lr = lr_preprocessor.transform(train_df.iloc[val_idx][features])
    X_te_lr = lr_preprocessor.transform(test_df[features])
    y_tr, y_va = y[train_idx], y[val_idx]
    
    lr_model = LogisticRegression(C=0.1, max_iter=1000, random_state=Config.RANDOM_STATE)
    lr_model.fit(X_tr_lr, y_tr)
    oof_lr[val_idx] = lr_model.predict_proba(X_va_lr)[:, 1]
    test_lr += lr_model.predict_proba(X_te_lr)[:, 1] / Config.N_SPLITS
    print(f"  [LR]   Fold {fold} ROC-AUC: {roc_auc_score(y_va, oof_lr[val_idx]):.5f}")
    
    # ------------------ 2. LightGBM Shallow ------------------
    X_tr_g, X_va_g = X_gbdt.iloc[train_idx], X_gbdt.iloc[val_idx]
    
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 4,
        'max_depth': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': Config.RANDOM_STATE + fold,
        'n_estimators': 1500,
        'verbose': -1,
        'n_jobs': -1
    }
    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_tr_g, y_tr,
        eval_set=[(X_va_g, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va_g)[:, 1]
    test_lgb += lgb_model.predict_proba(X_test_gbdt)[:, 1] / Config.N_SPLITS
    print(f"  [LGB]  Fold {fold} ROC-AUC: {roc_auc_score(y_va, oof_lgb[val_idx]):.5f}")
    
    # ------------------ 3. XGBoost GPU Hist ------------------
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        max_depth=3,
        learning_rate=0.05,
        n_estimators=1500,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        device=device,
        enable_categorical=True,
        early_stopping_rounds=50,
        random_state=Config.RANDOM_STATE + fold
    )
    xgb_model.fit(X_tr_g, y_tr, eval_set=[(X_va_g, y_va)], verbose=False)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va_g)[:, 1]
    test_xgb += xgb_model.predict_proba(X_test_gbdt)[:, 1] / Config.N_SPLITS
    print(f"  [XGB]  Fold {fold} ROC-AUC: {roc_auc_score(y_va, oof_xgb[val_idx]):.5f}")
    
    # ------------------ 4. CatBoost GPU ------------------
    cb_model = CatBoostClassifier(
        iterations=1200,
        learning_rate=0.05,
        depth=4,
        loss_function='Logloss',
        eval_metric='AUC',
        cat_features=cat_cols,
        task_type='GPU' if device == 'cuda' else 'CPU',
        early_stopping_rounds=50,
        random_seed=Config.RANDOM_STATE + fold,
        verbose=False
    )
    cb_model.fit(X_tr_g, y_tr, eval_set=(X_va_g, y_va), verbose=False)
    oof_cat[val_idx] = cb_model.predict_proba(X_va_g)[:, 1]
    test_cat += cb_model.predict_proba(X_test_gbdt)[:, 1] / Config.N_SPLITS
    print(f"  [CAT]  Fold {fold} ROC-AUC: {roc_auc_score(y_va, oof_cat[val_idx]):.5f}")
    
    print(f"  Fold {fold} finished in {time.time() - fold_start:.1f}s")

total_cv_time = time.time() - start_time
print("=" * 70)
print(f"All 4 models successfully trained across 5 folds in {total_cv_time:.1f}s!")

In [ ]:
model_names = ['Logistic Regression', 'LightGBM', 'XGBoost', 'CatBoost']
oof_dict = {
    'Logistic Regression': oof_lr,
    'LightGBM': oof_lgb,
    'XGBoost': oof_xgb,
    'CatBoost': oof_cat
}
test_dict = {
    'Logistic Regression': test_lr,
    'LightGBM': test_lgb,
    'XGBoost': test_xgb,
    'CatBoost': test_cat
}

summary_rows = []
for name in model_names:
    auc = roc_auc_score(y, oof_dict[name])
    pr_auc = average_precision_score(y, oof_dict[name])
    summary_rows.append({'Model': name, '5-Fold OOF ROC-AUC': f"{auc:.5f}", 'OOF PR-AUC': f"{pr_auc:.5f}"})

summary_df = pd.DataFrame(summary_rows)
print("\nStandalone Baseline Performance:")
print(summary_df.to_string(index=False))

# Save standalone OOF arrays
for name, arr in oof_dict.items():
    slug = name.lower().replace(' ', '_')
    np.save(Config.OUTPUT_DIR / f'oof_{slug}.npy', arr)

In [ ]:
# Compute Spearman rank correlation between OOF predictions
oof_matrix = pd.DataFrame({name: rankdata(oof_dict[name]) / len(y) for name in model_names})
corr_matrix = oof_matrix.corr(method='spearman')

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.4f', vmin=0.90, vmax=1.0)
plt.title('OOF Spearman Rank Correlation Matrix\n(High performance with diverse predictions = Strong ensemble)')
plt.tight_layout()
plt.savefig('model_diversity_correlation.png', dpi=150)
plt.show()

In [ ]:
# Normalize to percentile ranks for rank-averaging
rank_oofs = np.column_stack([rankdata(oof_dict[name]) / len(y) for name in model_names])
rank_tests = np.column_stack([rankdata(test_dict[name]) / len(test_df) for name in model_names])

# Strategy A: Simple Equal-Weighted Rank Average
blend_rank_oof = rank_oofs.mean(axis=1)
blend_rank_test = rank_tests.mean(axis=1)
auc_rank_avg = roc_auc_score(y, blend_rank_oof)
print(f"Strategy A [Equal Rank Average] OOF ROC-AUC: {auc_rank_avg:.5f}")

# Strategy B: Nelder-Mead Optimization on Rank Weights
def loss_func(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    comb = np.dot(rank_oofs, weights)
    return -roc_auc_score(y, comb)

init_weights = [0.1, 0.4, 0.4, 0.1]
res = minimize(loss_func, init_weights, method='Nelder-Mead', options={'maxiter': 500})
opt_weights = np.maximum(0, res.x)
opt_weights /= opt_weights.sum()

blend_opt_oof = np.dot(rank_oofs, opt_weights)
blend_opt_test = np.dot(rank_tests, opt_weights)
auc_opt = roc_auc_score(y, blend_opt_oof)

print(f"\nStrategy B [Nelder-Mead Optimal Rank Blend] OOF ROC-AUC: {auc_opt:.5f}")
print("Optimal Weights:")
for name, w in zip(model_names, opt_weights):
    print(f"  {name:20s}: {w:.4f} ({w*100:.1f}%)")

# Strategy C: Stacking Meta-Learner (Logistic Regression on Raw Probabilities)
stack_oof = np.zeros(n_train, dtype=np.float64)
stack_test = np.zeros(n_test, dtype=np.float64)

X_meta_train = np.column_stack([oof_dict[name] for name in model_names])
X_meta_test = np.column_stack([test_dict[name] for name in model_names])

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y), 1):
    meta_clf = LogisticRegression(C=1.0, random_state=Config.RANDOM_STATE)
    meta_clf.fit(X_meta_train[tr_idx], y[tr_idx])
    stack_oof[va_idx] = meta_clf.predict_proba(X_meta_train[va_idx])[:, 1]
    stack_test += meta_clf.predict_proba(X_meta_test)[:, 1] / Config.N_SPLITS

auc_stack = roc_auc_score(y, stack_oof)
print(f"\nStrategy C [Logistic Stacking Meta-Learner] OOF ROC-AUC: {auc_stack:.5f}")

In [ ]:
# Compare ROC curves
plt.figure(figsize=(10, 6))

for name in model_names:
    fpr, tpr, _ = roc_curve(y, oof_dict[name])
    plt.plot(fpr, tpr, lw=1.5, alpha=0.7, label=f"{name} (AUC: {roc_auc_score(y, oof_dict[name]):.5f})")

fpr_opt, tpr_opt, _ = roc_curve(y, blend_opt_oof)
plt.plot(fpr_opt, tpr_opt, color='black', lw=2.5, label=f"Optimal Blend (AUC: {auc_opt:.5f})")

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5)
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.02])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Ensemble vs. Individual Model Families (5-Fold Out-of-Fold ROC)')
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('ensemble_diagnostics.png', dpi=150)
plt.show()

In [ ]:
# Select the winning ensemble strategy based on highest 5-Fold OOF score
strategies = {
    'Strategy A (Equal Rank Blend)': (auc_rank_avg, blend_rank_test),
    'Strategy B (Optimal Rank Blend)': (auc_opt, blend_opt_test),
    'Strategy C (Stacking Meta-Model)': (auc_stack, stack_test)
}

best_strat_name = max(strategies.keys(), key=lambda k: strategies[k][0])
best_auc, final_preds = strategies[best_strat_name]

print("=" * 60)
print(f"WINNING ENSEMBLE STRATEGY: {best_strat_name}")
print(f"Out-of-Fold ROC-AUC:      {best_auc:.5f}")
print("=" * 60)

submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': final_preds
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'ensemble_submission.csv'
submission.to_csv(sub_path, index=False)
submission.to_csv(Config.OUTPUT_DIR / 'submission.csv', index=False)

print(f"\nSubmission verified and saved at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))